In [1]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler

plt.style.use('dark_background')
np.random.seed(42)
torch.manual_seed(42)

print("✅ Ready!")


✅ Ready!


In [2]:
# Load dataset
cancer = load_breast_cancer()
X = cancer.data
y = cancer.target

print(f"Dataset: Breast Cancer Wisconsin")
print(f"Total samples: {X.shape[0]}")
print(f"Features: {X.shape[1]}")
print(f"Classes: {len(np.unique(y))} (binary)")


Dataset: Breast Cancer Wisconsin
Total samples: 569
Features: 30
Classes: 2 (binary)


In [3]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)  # note that this is a single split and not a cross-validation

print(f"\nData split:")
print(f"  Training+Validation: {X_train_val.shape[0]} samples")
print(f"  External Test Set:   {X_test.shape[0]} samples")
print(f"\nTest set will NOT be used during hyperparameter tuning!")


Data split:
  Training+Validation: 455 samples
  External Test Set:   114 samples

Test set will NOT be used during hyperparameter tuning!


In [4]:
class SimpleNN(nn.Module):
    def __init__(self, input_size=30, hidden_size=15, num_classes=2):
        super(SimpleNN, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, num_classes)

    def forward(self, x):
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        return x

# Create model
model = SimpleNN()
print("Architecture:")
print(model)
print(f"\nParameters: {sum(p.numel() for p in model.parameters()):,}")


Architecture:
SimpleNN(
  (fc1): Linear(in_features=30, out_features=15, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=15, out_features=2, bias=True)
)

Parameters: 497


In [5]:
def train_model(model, X_train, y_train, X_val, y_val, lr, epochs=100):
    
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr)

    train_losses = []
    val_losses = []
    train_accs = []
    val_accs = []

    for epoch in range(epochs):
        # Training
        model.train()
        outputs = model(X_train)
        loss = criterion(outputs, y_train)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        _, train_pred = torch.max(outputs, 1)
        train_acc = (train_pred == y_train).float().mean().item()
        train_losses.append(loss.item())
        train_accs.append(train_acc)

        # Validation
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val)
            val_loss = criterion(val_outputs, y_val)
            _, val_pred = torch.max(val_outputs, 1)
            val_acc = (val_pred == y_val).float().mean().item()

        val_losses.append(val_loss.item())
        val_accs.append(val_acc)

    return train_losses, val_losses, train_accs, val_accs

print("Training function defined")


Training function defined


---
## Exercise: Repeat with 10-Fold CV

**Task:** Redo the cross-validation with **10 folds** instead of 5 for LR=0.001


In [6]:
# YOUR TASK: Perform 10-fold CV for LR=0.001
test_lr = 0.001

kFold = KFold(n_splits=10, shuffle=True, random_state=42)
fold_train_accs = []
fold_val_accs = []
for fold, (train_index, val_index) in enumerate(kFold.split(X_train_val),1):
    X_train_fold = X_train_val[train_index]
    y_train_fold = y_train_val[train_index]
    X_val_fold = X_train_val[val_index]
    y_val_fold = y_train_val[val_index]
    
    scaler = StandardScaler()
    X_train_fold = scaler.fit_transform(X_train_fold)
    X_val_fold = scaler.transform(X_val_fold)
    
    X_train_tensor = torch.FloatTensor(X_train_fold)
    y_train_tensor = torch.LongTensor(y_train_fold)
    X_val_tensor = torch.FloatTensor(X_val_fold)
    y_val_tensor = torch.LongTensor(y_val_fold)
    
    model_fold = SimpleNN()
    train_losses, val_losses, train_accs, val_accs = train_model(
        model_fold, X_train_tensor, y_train_tensor,
        X_val_tensor, y_val_tensor,
        lr=test_lr,epochs=100
    )
    fold_train_accs.append(train_accs[-1])
    fold_val_accs.append(val_accs[-1])
    
    mean_10fold = np.mean(fold_val_accs)
    std_10fold = np.std(fold_val_accs)
    
print(f"\n10-Fold CV Results (LR={test_lr}):")
print(f"  Mean Accuracy: {mean_10fold:.4f} ± {std_10fold:.4f}")



10-Fold CV Results (LR=0.001):
  Mean Accuracy: 0.9626 ± 0.0358
